# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfyuky/Starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np, pandas as pd, duckdb
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CSV = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
if not CSV.exists():
    CSV.parent.mkdir(parents=True, exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/yfyuky/Starter/main/data/raw/content_refresh_anonymized.csv", CSV)
con = duckdb.connect()
con.execute(f"CREATE VIEW t AS SELECT * FROM read_csv_auto('{CSV.as_posix()}')")
df = pd.read_csv(CSV)   # plain numpy dtypes: NaN comparisons evaluate to False
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

CONTRACT_USED = ["clicks_prev_30d", "sessions_prev_30d", "days_since_last_update",
                 "content_type", "search_volume", "word_count"]
LABEL_OR_EXCLUDED = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
                     "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier", "impression_tier"]
assert not set(CONTRACT_USED) & set(LABEL_OR_EXCLUDED)

vol_med = df["search_volume"].median()
wc_med = df["word_count"].median()
print("rows:", len(df), "| base rate (declining):", round(df.is_declining_label.mean(), 3))
print("median search_volume:", vol_med, "| median word_count:", wc_med)


rows: 30000 | base rate (declining): 0.542
median search_volume: 10.0 | median word_count: 2877.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
gate = (df.clicks_prev_30d > 0) & (df.sessions_prev_30d > 0)
p_stale = (df.days_since_last_update >= 91).astype(int) * 2
p_low   = (df.search_volume <= vol_med).astype(int)          # missing volume -> False
p_type  = (df.content_type != "feedly article").astype(int)
p_full  = (df.word_count >= wc_med).astype(int)              # missing words -> False
df["score"] = gate * (p_stale + p_low + p_type + p_full)

def codes(i):
    if not gate[i]:
        return "no_prior_traffic"
    c = ["had_prior_traffic"]
    if p_stale[i]: c.append("stale_90d_plus")
    if p_low[i]:   c.append("lower_demand_keyword")
    if p_type[i]:  c.append("standard_article_type")
    if p_full[i]:  c.append("fuller_content")
    return "|".join(c)

df["reason_codes"] = [codes(i) for i in df.index]
df["action"] = np.where(df.score >= 5, "review_first", np.where(df.score > 0, "review_for_refresh", "no_action"))

q = df.sort_values(["score", "days_since_last_update", "content_id"], ascending=[False, False, True]).reset_index(drop=True)
q["rank"] = q.index + 1
y = q.is_declining_label.values
base = y.mean()

def p_at_k(frame, k):
    return frame.is_declining_label.head(k).mean()

capped = q.groupby("client_id", sort=False).head(10).sort_values("rank")
rows = []
for k in (20, 50, 100, 500):
    rows.append({"K": k, "precision@K": round(p_at_k(q, k), 3), "precision@K (<=10 per client)": round(p_at_k(capped, k), 3),
                 "base rate": round(base, 3), "lift": round(p_at_k(q, k) / base, 2)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nscore distribution:\n", q.score.value_counts().sort_index(ascending=False).to_string())
print("ties at the top score:", int((q.score == q.score.max()).sum()), "pages")

out = ROOT / "work" / "outputs" / "baseline_action_score.csv"
out.parent.mkdir(parents=True, exist_ok=True)
q[["rank", "content_id", "client_id", "score", "reason_codes", "action"]].to_csv(out, index=False)
print("wrote", out.relative_to(ROOT), "rows:", len(q))


  K  precision@K  precision@K (<=10 per client)  base rate  lift
 20        0.950                          0.750      0.542  1.75
 50        0.620                          0.560      0.542  1.14
100        0.640                          0.540      0.542  1.18
500        0.608                          0.495      0.542  1.12

score distribution:
 score
5     1850
4     1695
3     2910
2     3190
1     1757
0    18598
ties at the top score: 1850 pages
wrote work/outputs/baseline_action_score.csv rows: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = q.head(20).copy()
def risk(r):
    n = []
    if r.sessions_prev_30d <= 2 or r.clicks_prev_30d <= 2: n.append("tiny prior traffic: could be noise")
    if r.days_since_last_update >= 170: n.append("very old update: may be dormant")
    if r.content_age_days < 120: n.append("young page: still ramping up")
    return "; ".join(n) if n else "client-level or seasonal dip would also fit"
top20["risk"] = top20.apply(risk, axis=1)
show = top20[["rank", "content_id", "score", "action", "days_since_last_update", "clicks_prev_30d",
              "sessions_prev_30d", "search_volume", "content_age_days", "risk"]].copy()
show["content_id"] = show["content_id"].astype(str).str[:8]
print(show.to_string(index=False))
print("\nclients represented in top 20:", top20.client_id.nunique())
print("share of top 20 from the single most common client:", round(top20.client_id.value_counts(normalize=True).iloc[0], 2))
print("hindsight (label is for CHECKING only): declined =", int(top20.is_declining_label.sum()), "of 20 | base rate", round(base, 3))


 rank content_id  score       action  days_since_last_update  clicks_prev_30d  sessions_prev_30d  search_volume  content_age_days                                                                risk
    1   content_      5 review_first                     194               22                 34            0.0               231                                     very old update: may be dormant
    2   content_      5 review_first                     194                1                  1            0.0               231 tiny prior traffic: could be noise; very old update: may be dormant
    3   content_      5 review_first                     194                1                  1            0.0               231 tiny prior traffic: could be noise; very old update: may be dormant
    4   content_      5 review_first                     194               28                 36            0.0               231                                     very old update: may be dormant
    5   co

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import inspect, re
# 1) the score uses only contract columns
used = {"clicks_prev_30d", "sessions_prev_30d", "days_since_last_update", "content_type", "search_volume", "word_count"}
forbidden = set(LABEL_OR_EXCLUDED) | {c for c in df.columns if "flag" in c.lower()}
assert not used & forbidden, "forbidden column feeds the score"

# 2) score is independent of the label: shuffle the label, recompute, same score
rng = np.random.RandomState(0)
shuffled = df.assign(trend_direction=rng.permutation(df.trend_direction.values))
g2 = (shuffled.clicks_prev_30d > 0) & (shuffled.sessions_prev_30d > 0)
s2 = g2 * ((shuffled.days_since_last_update >= 91).astype(int) * 2 + (shuffled.search_volume <= vol_med).astype(int)
           + (shuffled.content_type != "feedly article").astype(int) + (shuffled.word_count >= wc_med).astype(int))
assert (s2.values == df.score.values).all()
print("leakage checks passed: contract-only inputs, label shuffle leaves score unchanged")

# 3) the output file carries no names, URLs or queries
saved = pd.read_csv(out)
assert list(saved.columns) == ["rank", "content_id", "client_id", "score", "reason_codes", "action"]
assert not saved.astype(str).apply(lambda c: c.str.contains("http|www\\.", regex=True)).any().any()
print("CSV columns:", list(saved.columns), "| rows:", len(saved))

# 4) weak picks in the top 20, and by reason: is the top tier really better than the gated rest?
wk = top20[(top20.is_declining_label == 0)]
print("top-20 picks that did NOT decline:", len(wk), "of 20")
print("top-20 picks with tiny prior traffic (<=2 clicks or sessions):", int(((top20.clicks_prev_30d <= 2) | (top20.sessions_prev_30d <= 2)).sum()))
tier = q.groupby("action").is_declining_label.agg(["mean", "size"]).round(3)
print("\ndeclining rate by action tier (base rate", round(base, 3), "):\n", tier.to_string())
print("\nzero-prior-impression pages in the queued (score>0) set:", int(((q.score > 0) & (q.impressions_prev_30d == 0)).sum()), "(these can never be 'down')")


leakage checks passed: contract-only inputs, label shuffle leaves score unchanged
CSV columns: ['rank', 'content_id', 'client_id', 'score', 'reason_codes', 'action'] | rows: 30000
top-20 picks that did NOT decline: 1 of 20
top-20 picks with tiny prior traffic (<=2 clicks or sessions): 7

declining rate by action tier (base rate 0.542 ):
                      mean   size
action                          
no_action           0.519  18598
review_first        0.599   1850
review_for_refresh  0.577   9552

zero-prior-impression pages in the queued (score>0) set: 0 (these can never be 'down')


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.